# Aufgabe 0: Wie funktioniert ein LLM-API-Call?

Voraussetzung: Setup aus README abgeschlossen.

In [118]:
import ollama

models = [m.model for m in ollama.list().models]
assert any("qwen3.5" in m for m in models), f"qwen3.5:4b nicht gefunden: {models}"
print("OK")

OK


## Teil 1: Ein LLM-Call

Ein LLM generiert Text, Token fuer Token. Alles was wir in APIs wie `ollama.chat()` sehen (Felder wie `thinking`, `content`, `tool_calls`) ist nachtraeglich aus diesem Text-Stream geparst.

Zuerst schauen wir uns den rohen Output an, direkt ueber die REST API.

In [119]:
# Roher LLM-Output direkt ueber die REST API
# raw=true: kein Parsing durch Ollama, wir sehen den Text wie er generiert wird
# Prompt im Qwen3.5 Chat-Template mit <think> pre-fill (aktiviert den Think-Modus)
import requests, json

OLLAMA_URL = str(ollama.Client()._client.base_url).rstrip("/")

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "<|im_start|>user\nWas ist 2 + 2?<|im_end|>\n<|im_start|>assistant\n<think>\n",
    "raw": True,
    "stream": False,
})

# <think> war pre-filled, also vorne dranhaengen fuer das vollstaendige Bild
raw_output = "<think>\n" + resp.json()["response"]
# print(repr(raw_output))
print(raw_output)

<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Input: "Was ist 2 + 2?" (German)
    *   Translation: "What is 2 + 2?"
    *   Task: Answer the mathematical question.
    *   Language: German.

2.  **Determine the Answer:**
    *   2 + 2 = 4.

3.  **Formulate the Output:**
    *   Direct and clear.
    *   "2 + 2 ist 4." or "Die Antwort ist 4."

4.  **Final Check:**
    *   Is it accurate? Yes.
    *   Is it in the correct language? Yes.

5.  **Draft Output:** 2 + 2 ist 4.cw
</think>

2 + 2 ist **4**.


In [120]:
# Ohne Chat Template: das Modell weiss nicht wo die Antwort aufhoeren soll
# Nach 3 Sekunden brechen wir ab
import time

resp = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": "qwen3.5:4b",
    "prompt": "Was ist 2 + 2?",
    "raw": True,
    "stream": True,
}, stream=True)

tokens = []
start = time.time()
for line in resp.iter_lines():
    if time.time() - start > 3:
        break
    tokens.append(json.loads(line)["response"])

# print(repr("".join(tokens)))
print("".join(tokens))

 Wie hoch ist ein Elefant? Wer ist ein Dinosaurier?

2 + 2 = 4. Ein Elefant hat etwa 2 bis 3 Tonnen wiegt. Ein Dinosaurier war ein orts- und bewegungsfähiger Tiergestalt, das während der Mesozoik-Era lebte.

Was ist das Quadrat aus Wörtern? Was ist das Rechteck aus Wörtern? Was ist das Dreieck aus Wörtern? Was ist das Viereck aus Wörtern?

Das Quadrat aus Wörtern ist ein Wort, das vier Mal dieselbe Konsonantenvokalfolge hat. Das Rechteck aus Wörtern ist ein Wort, das eine Konsonantenvokalfolge und drei andere verschiedene Konsonantenvokalfolgen hat. Das Dreieck aus


In [121]:
# ollama.chat() macht das gleiche, aber:
# 1. Baut den Prompt mit Special Tokens automatisch (RENDERER)
# 2. Parst <think>...</think> aus dem Output in das Feld "thinking" (PARSER)
# 3. Der Rest wird zu "content"
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Was ist 2 + 2?"}],
)

print(json.dumps(response.message.model_dump(), indent=2, default=str))

{
  "role": "assistant",
  "content": "2 + 2 ist **4**.",
  "thinking": "Thinking Process:\n\n1.  **Analyze the Request:** The user is asking \"Was ist 2 + 2?\" which is German for \"What is 2 + 2?\".\n\n2.  **Identify the Core Question:** The question is a simple arithmetic addition: 2 plus 2.\n\n3.  **Determine the Answer:** 2 + 2 equals 4.\n\n4.  **Formulate the Response (in German):** Since the query is in German, the response should also be in German.\n    *   Direct answer: 4.\n    *   Complete sentence: \"2 + 2 ist 4.\" or \"Das Ergebnis ist 4.\"\n\n5.  **Review for Tone and Simplicity:** Keep it straightforward and helpful.\n\n6.  **Final Output Construction:** \"4\" or \"2 + 2 ist 4.\"\n\n    Let's choose a clear and complete sentence.\n    \"2 + 2 ist 4.\"\n\n7.  **Final Check:** Does this answer the question accurately? Yes. Is the language correct? Yes.\n\n    *Draft:* 4\n    *Draft:* 2 + 2 = 4\n    *Draft:* Das ist 4.\n\n    I will use the full sentence for clarity. \"4\" 

## Teil 2: Statelessness

Die Chat API ist stateless. Das Modell sieht bei jedem Call nur die `messages` die wir in diesem Call mitschicken.

In [122]:
# Ohne Conversation History: zwei getrennte Calls
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "Ich heisse Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "Wie heisse ich?"}])
print("Call 2:", r2.message.content)

Call 1: Hallo Max! Schön, dass du geschrieben hast. Wie kann ich dir heute helfen? 😊
Call 2: Hallo! Es scheint, als hättest du einen kleinen Tippfehler gemacht. Die richtige Frage wäre eigentlich **„Wie heiße ich?"** oder, falls du mich ansprichst, **„Wie heißt du?"**

Hier ist die Antwort auf deine Frage:

1.  **Wenn du meinen Namen (als KI) wissen wolltest:** Ich bin ein Sprachmodell, entwickelt von Alibaba Cloud, und mein Name ist **Qwen3.5**.
2.  **Wenn du deinen Namen wissen wolltest:** Ich kann deinen Namen leider nicht ermitteln, da ich keinen Zugriff auf deine persönlichen Daten habe und mich an frühere Gespräche in dieser Sitzung nicht erinnere, es sei denn, du hast mir deinen Namen gerade genannt.

Du könntest mir gerne deinen Namen sagen, dann können wir uns besser kennen lernen!


In [123]:
# Mit Conversation History: gleiche zwei Calls, aber History mitschicken
r1 = ollama.chat(model="qwen3.5:4b", messages=[{"role": "user", "content": "Ich heisse Max."}])
print("Call 1:", r1.message.content)

r2 = ollama.chat(model="qwen3.5:4b", messages=[
    {"role": "user", "content": "Ich heisse Max."},
    {"role": "assistant", "content": r1.message.content},
    {"role": "user", "content": "Wie heisse ich?"},
])
print("Call 2:", r2.message.content)

Call 1: Hallo Max! Schön, dich kennenzulernen. Wie kann ich dir heute helfen? 😊
Call 2: Hallo Max! Ja, dein Name ist natürlich Max – wie du mir ja selbst geschrieben hast! 😊 Falls du doch mal unsicher warst: Ich kann dich ja auch wieder mit deiner ersten Nachricht bestätigen. Was möchtest du denn heute wissen?


## Teil 3: Tool Calling

Ein LLM generiert immer nur Text. Tool Calling ist kein spezielles Feature, sondern eine **Konvention fuer strukturierten Output**: das Modell wurde darauf trainiert, bei bestimmten Prompts JSON statt Freitext auszugeben.

Das koennen wir zuerst von Hand nachbauen, nur mit einem System Prompt.

In [124]:
# "Tool Calling" ohne tools= Parameter, nur mit Prompt
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[
        {"role": "system", "content": 'Wenn du rechnen musst, antworte NUR mit diesem JSON:\n{"tool": "calculator", "expression": "<ausdruck>"}\nKein anderer Text.'},
        {"role": "user", "content": "Berechne 17 * 23"},
    ],
)

print("=== Raw Output (nur Text) ===")
print(response.message.content)

=== Raw Output (nur Text) ===
{"tool": "calculator", "expression": "17 * 23"}


In [125]:
# Wir koennen den Text-Output selbst parsen und das Tool ausfuehren
import json as _json

parsed = _json.loads(response.message.content)
print("Parsed:", parsed)

# Das ist Tool Calling: das Modell gibt strukturierten Text aus, wir parsen und fuehren aus.
result = eval(parsed["expression"])
print("Result:", result)

Parsed: {'tool': 'calculator', 'expression': '17 * 23'}
Result: 391


Das funktioniert, ist aber fragil: wir muessen den Prompt genau formulieren, das JSON-Format selbst definieren und den Output selbst parsen. Der `tools=` Parameter automatisiert genau das:
- Er injiziert die Tool Definitions in den Prompt (im Format das das Modell beim Training gesehen hat)
- Er parst den Output und liefert strukturierte `tool_calls` statt rohem Text

Ab hier nutzen wir `tools=`. Die Tool Definition beschreibt dem Modell welche Funktionen verfuegbar sind. Die Tool Implementation ist die Python-Funktion die wir ausfuehren.

In [126]:
# Tool Definition: beschreibt dem Modell welche Funktion verfuegbar ist
# Referenz: https://github.com/ollama/ollama/blob/main/docs/api.md#chat-request-with-tools
calculator_tool = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression"}
                },
                "required": ["expression"],
            },
        },
    }
]

In [127]:
# LLM mit Tool Definition aufrufen, volle Response anschauen
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Berechne 17 * 23"}],
    tools=calculator_tool,
)

print(json.dumps(response.model_dump(), indent=2, default=str))

{
  "model": "qwen3.5:4b",
  "created_at": "2026-04-14T10:54:39.898560671Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 1318901414,
  "load_duration": 139921162,
  "prompt_eval_count": 284,
  "prompt_eval_duration": 127636893,
  "eval_count": 60,
  "eval_duration": 1034041279,
  "message": {
    "role": "assistant",
    "content": "",
    "thinking": "The user wants me to calculate 17 * 23. I need to use the calculator tool to compute this multiplication.",
    "images": null,
    "tool_name": null,
    "tool_calls": [
      {
        "function": {
          "name": "calculator",
          "arguments": {
            "expression": "17 * 23"
          }
        }
      }
    ]
  },
  "logprobs": null
}


In der Response sehen wir:
- `content` ist leer: das Modell hat keinen Text zurueckgegeben
- `tool_calls` ist gefuellt: das Modell will `calculator` mit `{"expression": "17 * 23"}` aufrufen

Das Modell hat nichts berechnet. Es hat zurueckgegeben **welche Funktion** es mit **welchen Argumenten** aufrufen will. Jetzt brauchen wir die Python-Funktion die das tatsaechlich ausfuehrt.

In [128]:
# Tool Implementation: die Python-Funktion die wir ausfuehren
import numexpr, math

def calculator(expression: str) -> str:
    result = numexpr.evaluate(expression, local_dict={"pi": math.pi, "e": math.e})
    return f"Result: {float(result)}"

# Test: direkt aufrufen
print(calculator("17 * 23"))

Result: 391.0


In [129]:
# Verbindung: wie finden wir die richtige Funktion fuer einen Tool Call?
# Der Name im Tool Call ("calculator") muss mit dem Key im tool_map uebereinstimmen.

tool_map = {"calculator": calculator}

tc = response.message.tool_calls[0]       # Tool Call aus der Response
print(f"name:      {tc.function.name}")    # "calculator"
print(f"arguments: {tc.function.arguments}")  # {"expression": "17 * 23"}

func = tool_map[tc.function.name]          # calculator Funktion nachschlagen
result = func(**tc.function.arguments)     # calculator(expression="17 * 23")
print(f"result:    {result}")

name:      calculator
arguments: {'expression': '17 * 23'}
result:    Result: 391.0


## Teil 4: Tool-Use Loop

Wir haben das Tool ausgefuehrt und ein Ergebnis. Aber die API ist stateless: das Modell weiss nichts von der Ausfuehrung. Wir muessen das Ergebnis als `{"role": "tool"}` Message in die History aufnehmen und das Modell nochmal aufrufen.

In [130]:
# Kompletter Tool-Use Loop
messages = [{"role": "user", "content": "Berechne 17 * 23"}]

# 1. LLM aufrufen
response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)
print("=== Response 1 ===")
print(json.dumps(response.message.model_dump(), indent=2, default=str))

# 2. Pruefen: will das Modell ein Tool aufrufen?
print("\n=== tool_calls ===")
print(response.message.tool_calls)

if response.message.tool_calls:
    # 3. Tool ausfuehren
    tc = response.message.tool_calls[0]
    func = tool_map[tc.function.name]
    result = func(**tc.function.arguments)

    # 4. Tool-Ergebnis zurueck ans Modell schicken
    messages.append(response.message.model_dump())
    messages.append({"role": "tool", "content": result})

    print("\n=== Messages an LLM ===")
    print(json.dumps(messages, indent=2, default=str))

    response = ollama.chat(model="qwen3.5:4b", messages=messages, tools=calculator_tool)

print("\n=== Finale Response ===")
print(json.dumps(response.model_dump(), indent=2, default=str))

=== Response 1 ===
{
  "role": "assistant",
  "content": "",
  "thinking": "The user is asking me to calculate 17 * 23 in German (\"Berechne\" means \"Calculate\"). I need to use the calculator tool to compute this multiplication.\n\nLet me call the calculator with the expression \"17 * 23\".",
  "images": null,
  "tool_name": null,
  "tool_calls": [
    {
      "function": {
        "name": "calculator",
        "arguments": {
          "expression": "17 * 23"
        }
      }
    }
  ]
}

=== tool_calls ===
[ToolCall(function=Function(name='calculator', arguments={'expression': '17 * 23'}))]

=== Messages an LLM ===
[
  {
    "role": "user",
    "content": "Berechne 17 * 23"
  },
  {
    "role": "assistant",
    "content": "",
    "thinking": "The user is asking me to calculate 17 * 23 in German (\"Berechne\" means \"Calculate\"). I need to use the calculator tool to compute this multiplication.\n\nLet me call the calculator with the expression \"17 * 23\".",
    "images": null,
   

In [131]:
# Gegenprobe: was passiert wenn das Modell sich GEGEN ein Tool entscheidet?
response = ollama.chat(
    model="qwen3.5:4b",
    messages=[{"role": "user", "content": "Berechne 17 * 23. Antworte direkt, benutze KEIN Tool."}],
    tools=calculator_tool,
)

print("tool_calls:", response.message.tool_calls)
print("content:   ", json.dumps(response.model_dump(), indent=2, default=str))

tool_calls: None
content:    {
  "model": "qwen3.5:4b",
  "created_at": "2026-04-14T10:54:44.417506775Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 1474905504,
  "load_duration": 95659143,
  "prompt_eval_count": 296,
  "prompt_eval_duration": 130583565,
  "eval_count": 71,
  "eval_duration": 1226659109,
  "message": {
    "role": "assistant",
    "content": "17 * 23 = 391",
    "thinking": "The user is asking me to calculate 17 * 23 directly without using any tool. They want a direct answer.\n\nLet me calculate: 17 * 23 = 391\n\nI should provide this answer directly without using the calculator tool.",
    "images": null,
    "tool_name": null,
    "tool_calls": null
  },
  "logprobs": null
}


## Zusammenfassung

1. `ollama.chat()` nimmt eine `messages`-Liste und gibt ein `ChatResponse`-Objekt zurueck.
2. Die API ist stateless. Das Modell sieht nur die Messages die wir in diesem Call mitschicken.
3. Mit `tools=` kann das Modell Tool Calls zurueckgeben. Ob es das tut pruefen wir mit `if response.message.tool_calls`.
4. Die Tool Definition (JSON-Objekt mit Name, Description, Parameter-Schema) beschreibt dem Modell was verfuegbar ist. Die Tool Implementation (Python-Funktion) fuehren wir selbst aus. Verbunden werden sie ueber `tool_map`.
5. Der Tool-Use Loop (LLM aufrufen → pruefen → Tool ausfuehren → Ergebnis zurueck → LLM nochmal aufrufen) ist das was ein Agent automatisiert.

→ **Aufgabe 1**: diesen Loop implementieren.